In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, average_precision_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

# Change working directory to the notebook's directory
notebook_dir = os.path.dirname(os.path.abspath("frozen_linear_MCI.ipynb"))
os.chdir(notebook_dir)

class_counts = {0: 0.71, 1: 0.29}  # 100 samples of class 0, 20 samples of class 1
total = sum(class_counts.values())

# Compute weights inversely proportional to frequency
class_weight = {cls: total / (len(class_counts) * count) for cls, count in class_counts.items()}

# -------------------
# Config
# -------------------
n_folds = 5
train_pattern = "../../splits/adni/train_subject_list_adni_{}"  # train_subject_list_adni_0 ... _4
val_pattern   = "../../splits/adni/val_subject_list_adni_{}"    # val_subject_list_adni_0 ... _4
test_pattern  = "../../splits/adni/test_subject_list_adni_{}"   # test_subject_list_adni_0 ... _4
metadata_csv = "../../metadata/adni_metadata.csv"
feature_sets = ["../../latents/cls_adni_k8pcq4ai_300.npz"]
out_dir = "cv_classification_results"
os.makedirs(out_dir, exist_ok=True)

# -------------------
# Load metadata (contains label info)
# -------------------
df_meta = pd.read_csv(metadata_csv)
# Map subject id -> label (MCI -> 1, else 0) — adjust mapping if needed
class_map = dict(zip(df_meta["src_subject_id"].astype(str), df_meta["entry_research_group"].astype(str)))
class_map = {k: (1 if v == "MCI" else 0) for k, v in class_map.items()}

def save_predictions_csv(csv_path, subject_ids, y_true, y_score, y_pred):
    df = pd.DataFrame({
        "subject_id": subject_ids,
        "y_true": y_true.astype(int),
        "y_score": y_score.astype(float),
        "y_pred": y_pred.astype(int),
    })
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)

# -------------------
# Main loop: feature sets -> folds
# -------------------
rows = []  # accumulate per-fold rows for this feature file

for feat_file in feature_sets:
    features_dict = np.load(feat_file, allow_pickle=True)


    # Prepare plotting figure: 1 x n_folds subplots
    fig, axes = plt.subplots(1, n_folds, figsize=(4 * n_folds, 4), squeeze=False)
    axes = axes.ravel()

    for fold in range(n_folds):
        train_fname = train_pattern.format(fold)
        val_fname   = val_pattern.format(fold)
        test_fname  = test_pattern.format(fold)

        # Load IDs (dtype=str). If file missing, skip fold with warning.
        if not os.path.exists(train_fname) or not os.path.exists(val_fname):
            print(f"Warning: missing train/val files for fold {fold}: {train_fname} or {val_fname} not found. Skipping fold.")
            continue

        train_ids = np.loadtxt(train_fname, dtype=str)
        val_ids   = np.loadtxt(val_fname, dtype=str)
        test_ids  = np.loadtxt(test_fname, dtype=str)

        # Keep only IDs that exist in features and metadata (preserve order)
        tr_ok = [sid for sid in train_ids if sid in features_dict and sid.split('/')[0] in class_map]
        va_ok = [sid for sid in val_ids   if sid in features_dict and sid.split('/')[0] in class_map]
        test_ok = [sid for sid in test_ids  if sid in features_dict and sid.split('/')[0] in class_map]

        if len(tr_ok) == 0 or len(va_ok) == 0:
            print(f"Warning: fold {fold} for {feat_file} has empty train or val after filtering. Skipping fold.")
            continue

        # Build feature matrices
        X_train = np.vstack([features_dict[sid] for sid in tr_ok])
        X_val   = np.vstack([features_dict[sid] for sid in va_ok])
        X_test  = np.vstack([features_dict[sid] for sid in test_ok])
        X_train = np.nan_to_num(X_train, nan=0.0)
        X_val   = np.nan_to_num(X_val, nan=0.0)
        X_test  = np.nan_to_num(X_test, nan=0.0)

        # Build labels aligned to filtered IDs
        y_train = np.array([class_map[sid.split('/')[0]] for sid in tr_ok], dtype=np.int64)
        y_val   = np.array([class_map[sid.split('/')[0]] for sid in va_ok], dtype=np.int64)
        y_test  = np.array([class_map[sid.split('/')[0]] for sid in test_ok], dtype=np.int64)

        # If y_val has only one class, AUC/AUPRC cannot be computed -> handle later
        # Normalize features based on training set
        X_mean = X_train.mean(axis=0, keepdims=True)
        X_std  = X_train.std(axis=0, keepdims=True) + 1e-8
        X_train_norm = (X_train - X_mean) / X_std
        X_val_norm   = (X_val - X_mean) / X_std
        X_test_norm  = (X_test - X_mean) / X_std


        classes = np.unique(y_train)


        # Train logistic regression
        clf = LogisticRegression(max_iter=2000, class_weight=class_weight, solver="lbfgs")
        clf.fit(X_train_norm, y_train)

        # Predict
        y_pred = clf.predict(X_val_norm)
        # predicted scores for class 1 (positive)
        if hasattr(clf, "predict_proba"):
            y_score = clf.predict_proba(X_val_norm)[:, 1]
        else:
            # if no predict_proba, use decision_function and minmax-scale
            try:
                scores = clf.decision_function(X_val_norm)
                # scale to [0,1]
                y_score = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
            except Exception:
                # fallback to predicted labels as scores
                y_score = y_pred.astype(float)

        # Metrics (guard AUC/AUPRC when single-class in y_val)
        bal_acc = balanced_accuracy_score(y_val, y_pred)

        try:
            auprc = average_precision_score(y_val, y_score)
        except Exception:
            auprc = np.nan

        try:
            auc = roc_auc_score(y_val, y_score)
        except Exception:
            auc = np.nan

        # Predictions and scores for test
        y_pred_test = clf.predict(X_test_norm)
        y_score_test = clf.predict_proba(X_test_norm)[:, 1]

        csv_path = f'CV_MCI/frozen_linear_val_{fold}.csv'
        save_predictions_csv(csv_path, va_ok, y_val, y_score, y_pred)
        csv_path = f'CV_MCI/frozen_linear_test_{fold}.csv'
        save_predictions_csv(csv_path, test_ok, y_test, y_score_test, y_pred_test)

        rows.append({
            "feature_file": os.path.basename(feat_file),
            "fold": fold,
            "n_train": len(y_train),
            "n_val": len(y_val),
            "BalancedAccuracy": bal_acc,
            "AUPRC": auprc,
            "AUROC": auc
        })

        # Plot jittered ground-truth vs predicted labels for this fold
        ax = axes[fold]
        jitter_gt = y_val + np.random.uniform(-0.05, 0.05, size=len(y_val))
        jitter_pred = y_pred + np.random.uniform(-0.05, 0.05, size=len(y_pred))
        ax.scatter(jitter_gt, jitter_pred, alpha=0.6)
        ax.plot([-.5, 1.5], [-.5, 1.5], 'r--')
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Control", "MCI"])
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["Control", "MCI"])
        ax.set_xlabel("Ground Truth")
        ax.set_ylabel("Predicted")
        ax.set_title(f"fold {fold}\nBA={bal_acc:.3f} AUPRC={np.nan_to_num(auprc):.3f} AUROC={np.nan_to_num(auc):.3f}")

    # End folds loop

# Convert rows to DataFrame and save
df_res = pd.DataFrame(rows)
csv_out = os.path.join(out_dir, f"cv_classification_table_{os.path.basename(feat_file)}.csv")
df_res.to_csv(csv_out, index=False)
print(f"\nPer-fold results for {feat_file}:")
print(df_res)

# Finalize and show plot
fig.suptitle(f"Classification predictions per fold — {os.path.basename(feat_file)}", fontsize=12)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("Done. Per-fold CSVs and summaries (when available) saved to:", out_dir)



In [ ]:
# Aggregate df_res by feature_file and compute mean ± std of rho_norm
if 'df_res' in globals() and (isinstance(df_res, (pd.DataFrame)) and not df_res.empty):
    agg = df_res.groupby('feature_file')['BalancedAccuracy'].agg(['mean','std']).reset_index()
    agg['acc_mean_std'] = agg.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1)
    agg = agg.rename(columns={'mean': 'rho_mean', 'std': 'rho_std'})[['feature_file','rho_mean','rho_std','acc_mean_std']]
    print('Mean ± std of rho_norm by feature_file:')
    display(agg[['feature_file','acc_mean_std']])
    os.makedirs(out_dir, exist_ok=True)
    #out_csv = os.path.join(out_dir, 'rho_norm_summary_by_feature.csv')
    #agg.to_csv(out_csv, index=False)
    #print(f'Saved summary CSV: {out_csv}')
else:
    print('df_res not found or empty; run the previous cells to compute df_res before aggregating.')